# 03 · Explainability

SHAP explanations for the best tree model and term-level explanations for the review-text model (PLAN.md section 7). Run after `02_models` (it needs `main_comparison.csv` and the out-of-fold scores). This also saves the models used by the Streamlit demo.

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, HTML, display, Markdown
from loanshield.common import ROOT, path, cfg
os.chdir(ROOT)                      # all paths are relative to the repo root
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_colwidth", 60, "display.max_columns", 30)
print("repo:", ROOT)

from loanshield.explain import shap_report
N_ITER = int(os.environ.get("LOANSHIELD_N_ITER", 30))
card = {**shap_report.tabular_report(N_ITER), **shap_report.text_report()}
card

## Global feature importance

In [ ]:
display(Image(str(path("results", "figures", "shap_summary_beeswarm.png"))))
display(Image(str(path("results", "figures", "shap_bar_top20.png"))))
imp = pd.read_csv(path("results", "tables", "shap_importance.csv"))
imp.head(20)

## Dependence plots (top 3 features)

In [ ]:
for f in sorted(path("results", "figures").glob("shap_dependence_*.png")):
    display(Markdown(f"**{f.stem.removeprefix('shap_dependence_')}**")); display(Image(str(f)))

## Per-app explanations
The model's top 3 reasons in each direction, in plain language. Risk scores are **out-of-fold** (the app was not in the training fold). The highest-scoring apps:

In [ ]:
ex = pd.read_csv(path("results", "tables", "app_explanations.csv"))
display(ex.head(10)[["title", "label", "risk_score_oof", "warning_1", "warning_2", "warning_3"]])
print("Verified apps the model scores highest (false-positive candidates):")
display(ex[ex.label == "verified"].head(5)[["title", "risk_score_oof", "warning_1", "warning_2"]])

## Review-text model
Highest-weight n-grams per class (TF-IDF + logistic regression on all labelled apps' reviews). Check whether the "unverified" terms are risk language or just app and product names. Names mean the model is memorising apps.

In [ ]:
display(Image(str(path("results", "figures", "shap_text_top_terms.png"))))
display(pd.read_csv(path("results", "tables", "text_top_terms.csv")).head(15))

In [ ]:
display(HTML(path("results", "figures", "text_examples.html").read_text(encoding="utf-8")))